<a href="https://www.kaggle.com/code/ravikushwah095/exercise-classification-by-lstm?scriptVersionId=337189445" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # print(os.path.join(dirname, filename))
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np 
import pandas as pd

X_correct = np.load("/kaggle/input/datasets/mohamadashrafsalama/pushup/labels/correct.npy")
X_incorrect = np.load("/kaggle/input/datasets/mohamadashrafsalama/pushup/labels/incorrect.npy")

X_correct.shape,X_incorrect.shape

In [ ]:
y_correct = np.ones(50)
y_incorrect = np.zeros(50)
y_correct.shape,y_incorrect.shape


In [ ]:
import os, random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
X = np.concatenate([X_correct,X_incorrect],axis=0)
y = np.concatenate([y_correct,y_incorrect],axis=0)

X.shape,y.shape

In [ ]:
X_motion = np.diff(X, axis=1)

print(X_motion.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_motion, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

mean = X_train.mean(axis=(0,1), keepdims=True)
std = X_train.std(axis=(0,1), keepdims=True) + 1e-8

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
# mean = X_train.mean(axis=(0,1), keepdims=True)
# std = X_train.std(axis=(0,1), keepdims=True) + 1e-8

# X_train_n = (X_train - mean) / std
# X_test_n  = (X_test - mean) / std

# print(X_train_n.mean(), X_train_n.std())

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(150, 66)),
    Dropout(0.3),

    LSTM(32),
    Dropout(0.3),

    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense

# model = Sequential([
#     LSTM(32, input_shape=(150, 66)),
#     Dense(1, activation='sigmoid')
# ])

# model.compile(
#     optimizer='adam',
#     loss='binary_crossentropy',
#     metrics=['accuracy']
# )

# history = model.fit(
#     X_train_n, y_train,
#     epochs=80,
#     batch_size=8,
#     validation_data=(X_test_n, y_test),
#     verbose=1
# )

# loss, acc = model.evaluate(X_test_n, y_test, verbose=0)
# print('Test accuracy:', acc)

In [ ]:
# history = model.fit(
#     X_train, y_train,
#     epochs=30,
#     batch_size=8,
#     validation_split=0.2
# )
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_accuracy',   # validation accuracy dekho
    mode='max',               # maximum value chahiye
    patience=5,               # 5 epochs tak improvement na ho to stop
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=8,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    LSTM(32, input_shape=(149, 66)),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

print('Best val acc:', max(history.history['val_accuracy']))

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {acc:.4f}')

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print('Best restored accuracy:', acc)

In [ ]:
print('Best val acc:', max(history.history['val_accuracy']))
print('Best train acc:', max(history.history['accuracy']))


In [ ]:
import matplotlib.pyplot as plt

# feature 0 (e.g., nose x-coordinate)
plt.plot(X_correct[0, :, 0], label='Correct')
plt.plot(X_incorrect[0, :, 0], label='Incorrect')

plt.legend()
plt.title('Feature 0 over 150 frames')
plt.show()

In [ ]:
sample = X_test[0:1]   # shape (1,150,66)

pred = model.predict(sample)

print('Probability:', pred[0][0])

if pred[0][0] > 0.5:
    print('Correct Push-up')
else:
    print('Incorrect Push-up')

In [ ]:
print(history.history.keys())

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train')

if 'val_accuracy' in history.history:
    plt.plot(history.history['val_accuracy'], label='val')

plt.legend()
plt.show()

In [ ]:
probs = model.predict(X_test).reshape(-1)

print(np.round(probs, 3))

In [ ]:
pred = (probs > 0.6).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score

print('Acc:', accuracy_score(y_test, pred))
print('Pred count:', np.bincount(pred))

Checking Performance using GRU

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

gru_model = Sequential([
    GRU(64, return_sequences=True,
        input_shape=(150, 66)),
    Dropout(0.3),

    GRU(32),
    Dropout(0.3),

    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

gru_model.summary()

In [ ]:
gru_history = gru_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

In [ ]:
lstm_loss, lstm_acc = model.evaluate(X_test, y_test, verbose=0)
gru_loss, gru_acc = gru_model.evaluate(X_test, y_test, verbose=0)

print(f'LSTM Accuracy: {lstm_acc:.4f}')
print(f'GRU  Accuracy: {gru_acc:.4f}')

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['val_accuracy'], label='LSTM')
plt.plot(gru_history.history['val_accuracy'], label='GRU')

plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
pred = (model.predict(X_test) > 0.5).astype(int)

print('Predictions:', pred.T)
print('True labels:', y_test.astype(int))

Check WIth KFold

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_motion, y), 1):

    X_train = X_motion[train_idx]
    X_test  = X_motion[test_idx]
    y_train = y[train_idx]
    y_test  = y[test_idx]

    # normalize
    mean = X_train.mean(axis=(0,1), keepdims=True)
    std = X_train.std(axis=(0,1), keepdims=True) + 1e-8

    X_train = (X_train - mean) / std
    X_test  = (X_test - mean) / std

    model = Sequential([
        LSTM(32, input_shape=(149,66)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.fit(X_train, y_train,
              epochs=40,
              batch_size=8,
              verbose=0)

    _, acc = model.evaluate(X_test, y_test, verbose=0)

    scores.append(acc)
    print(f'Fold {fold}: {acc:.3f}')

print('Mean accuracy:', np.mean(scores))
print('Std accuracy :', np.std(scores))

## Push-Up Form Classification using LSTM and MediaPipe

### Project Summary

Developed a deep learning pipeline to classify push-up exercises as **Correct** or **Incorrect** using pose sequences extracted from videos with **MediaPipe Pose**. The project focused on temporal sequence modeling and comparative evaluation of recurrent neural network architectures.

### Dataset

* **100 pose sequences** (50 correct, 50 incorrect)
* **150 frames per sequence**
* **66 features per frame** (33 body landmarks × x-y coordinates)

### Experimental Approach

* Loaded pose sequences from `.npy` files and encoded labels for binary classification.
* Applied **z-score normalization** for feature standardization.
* Performed **feature engineering using frame-difference motion features (`np.diff`)** to capture movement dynamics.
* Trained and compared **LSTM and GRU** models using identical preprocessing and training settings.
* Evaluated performance using:

  * Train/validation accuracy
  * Confusion matrix and prediction analysis
  * Threshold-based probability analysis
  * **5-fold stratified cross-validation** for reliable performance estimation
* Used **EarlyStopping with best-weight restoration** to reduce overfitting on the small dataset.

### Final Model

The **LSTM model** provided the best overall performance and stability compared with the GRU model and other baseline experiments.

### Results

* **Best validation accuracy:** 70%
* **5-fold cross-validation accuracy:** **64.0% ± 3.7%**
* The model consistently outperformed the **50% random baseline**, indicating successful learning of temporal pose patterns.

### Conclusion

The project demonstrates an end-to-end workflow for **pose-based exercise assessment**, including preprocessing, motion-feature engineering, recurrent model comparison, overfitting analysis, and cross-validation. The final LSTM model showed the best generalization performance and can serve as a foundation for AI-powered fitness coaching or rehabilitation systems.
